# NB_lab_test_trends

Builds `adb_medilake.gold.lab_test_trends` per the STTM *Silver to Gold (CareSync Health Network)* v1.0, sheet `lab_test_trends`.

* **Grain:** one row per test type (test_name) per calendar month
* **Driving table:** `adb_medilake.silver.lab_results` grouped by test_name, year/month of test_date
* **Joins:** LEFT JOIN `silver.calendar` on date_key = yyyyMMdd(first day of month)
* **Window:** LAG(abnormal_rate_pct) partitioned by test_name ordered by year, month
* **Load:** FULL rebuild with CREATE OR REPLACE TABLE ... AS SELECT

Implementation note: `silver.lab_results` contains exact duplicate `lab_result_id` rows, so lab rows are de-duplicated on `lab_result_id` (latest `update_timestamp`) before aggregating.

In [0]:
%sql
-- Full rebuild of gold.lab_test_trends (one row per test_name per month)
CREATE OR REPLACE TABLE adb_medilake.gold.lab_test_trends
COMMENT 'Monthly lab test volume and abnormal/critical rates with month-over-month trend (STTM silver to gold v1.0)'
AS
WITH lab AS (
  -- De-duplicate lab results to one row per lab_result_id
  SELECT *
  FROM adb_medilake.silver.lab_results
  QUALIFY row_number() OVER (PARTITION BY lab_result_id ORDER BY update_timestamp DESC) = 1
),
monthly AS (
  SELECT
    test_name,
    year(test_date)                                  AS year,
    month(test_date)                                 AS month,
    CAST(date_trunc('MONTH', MIN(test_date)) AS DATE) AS month_start,
    COUNT(*)                                         AS total_tests,
    COUNT_IF(result_status = 'Abnormal')             AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')             AS critical_test_count
  FROM lab
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT
    *,
    CAST(abnormal_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(critical_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly
),
trend AS (
  SELECT
    *,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates
)
SELECT
  t.test_name,
  CAST(t.year AS INT)                                                    AS year,
  CAST(t.month AS INT)                                                   AS month,
  c.month_name,
  c.quarter_name,
  CAST(t.total_tests AS INT)                                             AS total_tests,
  CAST(t.abnormal_test_count AS INT)                                     AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                                     AS critical_test_count,
  t.abnormal_rate_pct,
  t.critical_rate_pct,
  CAST(t.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                   AS prev_month_abnormal_rate_pct,
  CAST(t.abnormal_rate_pct - t.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM trend t
LEFT JOIN adb_medilake.silver.calendar c
  ON c.date_key = CAST(date_format(t.month_start, 'yyyyMMdd') AS INT);

In [0]:
%sql
-- Validation: grain uniqueness, test totals reconcile to de-duplicated silver, calendar coverage
SELECT
  count(*)                                    AS total_rows,
  count(DISTINCT test_name, year, month)      AS distinct_test_months,
  sum(total_tests)                            AS total_tests,
  (SELECT count(DISTINCT lab_result_id) FROM adb_medilake.silver.lab_results) AS source_tests,
  count_if(month_name IS NULL)                AS missing_calendar,
  count_if(prev_month_abnormal_rate_pct IS NULL) AS first_months
FROM adb_medilake.gold.lab_test_trends;

In [0]:
%sql
select * from adb_medilake.gold.lab_test_trends
order by test_name, year, month
limit 10

In [0]:
# Return the record count of the rebuilt gold table as the notebook exit value
updated_df = spark.table("adb_medilake.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))